In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments

# Load pretrained GPT-2 model and tokenizer
model = AutoModelForCausalLM.from_pretrained('gpt2')
tokenizer = AutoTokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token  # ensure pad token is defined

# Example training data: a list of dicts with "prompt" and "response"
train_data = [
    {"prompt": "User: How do I bake a cake?\nAssistant:", 
     "response": " Sure! To bake a cake, first preheat your oven to 350°F... [detailed steps]"},
    {"prompt": "User: What is the capital of France?\nAssistant:", 
     "response": " The capital of France is Paris."},
    # ... (more examples)
]

train_encodings = {"input_ids": [], "attention_mask": [], "labels": []}

# Convert to a format suitable for training (concatenate prompt and response)
max_len = 64
for ex in train_data:
    prompt = ex["prompt"]
    response = ex["response"]

    # Tokenize the training data
    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    response_ids = tokenizer(response, add_special_tokens=False)["input_ids"]
    
    input_ids = prompt_ids + response_ids
    attention_mask = [1] * len(input_ids)

    # Mask the prompt so the model learns the response
    labels = [-100] * len(prompt_ids) + response_ids

    padding = max_len - len(input_ids)
    if max_len > len(input_ids):
        attention_mask += [0] * padding
        labels += [-100] * padding
        input_ids += [tokenizer.pad_token_id] * padding
    else:
        input_ids = input_ids[:max_len]
        attention_mask = attention_mask[:max_len]
        labels = labels[:max_len]

    train_encodings["input_ids"].append(input_ids)
    train_encodings["attention_mask"].append(attention_mask)
    train_encodings["labels"].append(labels)

In [ ]:
import torch
from torch.utils.data import Dataset

class TextDataset(Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        # Labels are the same as input_ids for causal LM
        return {
            "input_ids": torch.tensor(self.encodings["input_ids"][idx]),
            "attention_mask": torch.tensor(self.encodings["attention_mask"][idx]),
            "labels": torch.tensor(self.encodings["labels"][idx])
        }

train_dataset = TextDataset(train_encodings)
print(len(train_dataset))

In [ ]:
# Define training arguments (small training for demo purposes)
training_args = TrainingArguments(
    output_dir="sft-model",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    logging_steps=10,
    learning_rate=5e-5,
    fp16=False,
    report_to=None  # no integrated logging
)

# Create a Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset  # in practice, prepare a proper Dataset object
)

# Fine-tune the model (this will run for a few epochs)
trainer.train()

In [ ]:
from transformers import pipeline

# Use a sentiment analysis pipeline as a proxy reward model
sentiment_pipe = pipeline('sentiment-analysis')

def compute_reward(response_text):
    """Compute a reward score given the response text."""
    result = sentiment_pipe(response_text, truncation=True)[0]  # analyze sentiment
    if result['label'] == 'POSITIVE':
        # reward is +1 times confidence if positive
        return result['score']  
    else:
        # reward is -1 times confidence if negative
        return -result['score']

In [ ]:
from trl import AutoModelForCausalLMWithValueHead, PPOTrainer, PPOConfig, create_reference_model

# Load the fine-tuned model into a value-head model for PPO
ppo_model = AutoModelForCausalLMWithValueHead.from_pretrained("sft-model")  # using the SFT model weights
ppo_model.to("mps")
tokenizer.pad_token = tokenizer.eos_token  # ensure tokenizer has a pad token set
# Create a reference model (for KL penalty) – usually a copy of initial policy
ref_model = create_reference_model(ppo_model)

# Set up PPO configuration
ppo_config = PPOConfig(
    batch_size=1,  # how many prompts per PPO update
    mini_batch_size=1,
    optimize_cuda_cache=False
)
ppo_trainer = PPOTrainer(config=ppo_config, model=ppo_model, ref_model=ref_model, tokenizer=tokenizer)

In [ ]:
# Sample prompt for the model to respond to
prompt = "User: I don't think my job interview went well.\nAssistant:"

# Tokenize prompt and move to device
input_ids = [tokenizer(prompt, return_tensors="pt").input_ids[0].to("mps")]

# Generate a response from the current policy model
gen_output = ppo_trainer.generate(input_ids, max_new_tokens=50, pad_token_id=tokenizer.eos_token_id)
response_ids = gen_output[0]
response_text = tokenizer.decode(response_ids, skip_special_tokens=True)
print("Model response before RLHF:", response_text)

# Compute reward using our proxy reward function
reward_score = compute_reward(response_text)
reward_tensor = [torch.tensor(reward_score, dtype=torch.float32, device="mps")]

# Run a PPO optimization step
train_stats = ppo_trainer.step(input_ids, [response_ids], reward_tensor)

In [ ]:
# Generate from model after PPO update
input_ids_tensor = torch.stack(input_ids).to("mps")
attention_mask_tensor = (input_ids_tensor != tokenizer.pad_token_id).long()
new_response_ids = ppo_model.generate(
    input_ids_tensor,
    attention_mask=attention_mask_tensor,
    max_new_tokens=50,
    pad_token_id=tokenizer.eos_token_id,
    top_k=50,
    top_p=0.95)
new_response_text = tokenizer.decode(new_response_ids[0], skip_special_tokens=True)
print("Model response after one PPO step:", new_response_text)

In [ ]:
from more_itertools import chunked

batch_size = 2
prompts = ["User: How was your day?\nAssistant:",
           "User: I don't like my job.\nAssistant:",
           "User: Tell me a joke.\nAssistant:",
           "User: I need advice on studying.\nAssistant:",
           "User: What's the weather like?\nAssistant:"]

for epoch in range(5):
    for prompt in prompts:
        encoding = tokenizer(prompt, return_tensors='pt')

        query = encoding.input_ids[0].to("mps")

        attention_mask = encoding.attention_mask.to("mps")

        response = ppo_trainer.generate(
            query,
            attention_mask=attention_mask,
            max_new_tokens=50,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=True,
            top_k=50,
            top_p=0.95)

        response = response.squeeze(0) if response.dim() == 2 else response
    
        text = tokenizer.decode(response, skip_special_tokens=True)
        reward = torch.tensor(compute_reward(text), dtype=torch.float32)

        train_states = ppo_trainer.step(
            [query.cpu()],
            [response.cpu()],
            [reward.cpu()]
        )

        print("Prompt:", prompt)
        print("Response:", text)
        print("Reward:", reward.item())
        print("PPO step done\n")